# R2V Deployment Notebook — Stable Diffusion → Hunyuan3D-2.1 → optional Texture

**Workflow:** `prompt / voice_text → Stable Diffusion image → Hunyuan3D-2.1 mesh → optional Hunyuan3D-Paint PBR texture → GLB`

This is a rebuilt, deployment-ready version. Run the cells top to bottom.

### What changed vs. the old notebook
- **Fixed a guaranteed cold-start crash.** The old default model `runwayml/stable-diffusion-v1-5` was removed from HuggingFace in 2024 and now 404s. This uses the maintained mirror `stable-diffusion-v1-5/stable-diffusion-v1-5` (and `AutoPipelineForText2Image`, so you can drop in SDXL too).
- **Async job queue instead of a 2-hour blocking request.** A request used to hold the HTTP connection open for the entire (multi-minute to multi-hour) generation, which times out behind any real proxy / load balancer / chatbot backend. Now: `POST /generate` returns a `job_id` instantly, and you poll `GET /jobs/{job_id}`.
- **Split tiers:** a tiny fast **CPU web tier** (FastAPI) + a heavy **GPU tier** (`modal.Cls`) that loads models once on container start (`@modal.enter`) instead of on first request.
- **The app code is written with `%%writefile`** (next cell) so it's fully readable and editable here — no more escaped one-line string.
- Robustness: graceful rembg fallback, VRAM cleanup between stages, configurable CORS, optional bearer-token auth, path-traversal-safe downloads, and an optional `prewarm` step to pre-download weights.

### Endpoints
- `GET /health` — cheap liveness check (no GPU)
- `POST /generate` — text / voice prompt → `{job_id}`
- `POST /image-to-3d` — uploaded image → `{job_id}`
- `GET /jobs/{job_id}` — status + result URLs
- `GET /download/{job_id}/{filename}` — GLB / image download


## 1. Prerequisites (run once, locally)

```bash
pip install modal
modal setup
```

After deployment, point your R2V backend at the printed URL:

```env
MODAL_API_URL=https://YOUR-WORKSPACE--r2v-sd-hunyuan3d-api.modal.run
MODAL_PROMPT_TO_3D_PATH=/generate
MODAL_IMAGE_TO_3D_PATH=/image-to-3d
```


In [2]:
# 0. Install / update local tools.
# Run this once in the notebook environment.
%uv pip install -U modal requests
!modal --version

# If Modal is not logged in yet, run this once manually:
# !modal setup


Using Python 3.12.6 environment at: /usr/local
Resolved 32 packages in 622ms
Audited 32 packages in 0.25ms
Note: you may need to restart the kernel to use updated packages.
modal client version: 1.5.0


In [1]:
%%writefile r2v_modal_app.py
"""r2v_modal_app.py — Deployment-ready Modal API for R2V.

Workflow:
    prompt / voice_text -> Stable Diffusion condition image
                        -> Hunyuan3D-2.1 mesh
                        -> optional Hunyuan3D-Paint texture -> GLB

Deploy:
    modal deploy r2v_modal_app.py

Optional cache warm-up:
    modal run r2v_modal_app.py
"""

import json
import modal

APP_NAME = "r2v-sd-hunyuan3d"
MINUTES = 60
HOURS = 60 * MINUTES

CACHE_DIR = "/cache"
OUTPUT_DIR = "/outputs"
REPO_DIR = "/opt/Hunyuan3D-2.1"

GPU_TYPE = "H100"
DEFAULT_T2I_MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"

TORCH_LIB = "/usr/local/lib/python3.10/site-packages/torch/lib"
CUDA_LIB = "/usr/local/cuda/lib64"
LD_PATH = f"{TORCH_LIB}:{CUDA_LIB}:/usr/local/lib"

cache_volume = modal.Volume.from_name("r2v-hf-cache", create_if_missing=True)
outputs_volume = modal.Volume.from_name("r2v-generated-assets", create_if_missing=True)
job_store = modal.Dict.from_name("r2v-job-store", create_if_missing=True)

# Optional:
# modal secret create r2v-secrets R2V_API_TOKEN=yourtoken HF_TOKEN=hf_xxx
# secret = modal.Secret.from_name("r2v-secrets")


gpu_image = (
    modal.Image.from_registry("nvidia/cuda:12.4.1-devel-ubuntu22.04", add_python="3.10")
    .apt_install(
        "git",
        "wget",
        "curl",
        "build-essential",
        "gcc",
        "g++",
        "clang",
        "cmake",
        "ninja-build",
        "ffmpeg",
        "libgl1",
        "libglib2.0-0",
        "libxrender1",
        "libxext6",
        "libsm6",
    )
    .run_commands(
        "python -m pip install --upgrade pip setuptools wheel",
        "pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 "
        "--index-url https://download.pytorch.org/whl/cu124",
        "python -c \"import torch; print('torch ok:', torch.__version__, 'cuda:', torch.version.cuda)\"",
    )
    .pip_install(
        "accelerate==0.33.0",
        "diffusers==0.31.0",
        "huggingface-hub==0.36.0",
        "pydantic==2.10.6",
        "safetensors==0.4.5",
        "sentencepiece==0.2.0",
        "transformers~=4.44.0",
        "pillow==10.4.0",
        "trimesh==4.5.3",
        "opencv-python-headless==4.10.0.84",
        "numpy<2",
        "rembg[gpu]==2.0.59",
        "onnxruntime-gpu==1.18.1",
        "hf_transfer==0.1.8",
    )
    .run_commands(
        f"git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git {REPO_DIR}",

        f"cd {REPO_DIR} && grep -ivE "
        "'^[[:space:]]*(bpy([[:space:]]|[=<>!~]|$)|"
        "torch([[:space:]]|[=<>!~]|$)|"
        "torchvision([[:space:]]|[=<>!~]|$)|"
        "torchaudio([[:space:]]|[=<>!~]|$)|"
        "onnxruntime([[:space:]]|[=<>!~]|$)|"
        "onnxruntime-gpu([[:space:]]|[=<>!~]|$)|"
        "opencv-python([[:space:]]|[=<>!~]|$)|"
        "opencv-python-headless([[:space:]]|[=<>!~]|$)|"
        "gradio([[:space:]]|[=<>!~]|$)|"
        "fastapi([[:space:]]|[=<>!~]|$)|"
        "uvicorn([[:space:]]|[=<>!~]|$)|"
        "deepspeed([[:space:]]|[=<>!~]|$)|"
        "--extra-index-url|--index-url|--find-links|-i[[:space:]]|-f[[:space:]])' "
        "requirements.txt > requirements.modal.txt",

        f"cd {REPO_DIR} && pip install --timeout 300 --retries 10 -r requirements.modal.txt",

        # Hunyuan Paint / RealESRGAN / BasicSR still import pkg_resources.
        # pkg_resources is provided by setuptools; pin a compatible version after
        # the Hunyuan requirements because that install can change packages.
        "pip install --force-reinstall setuptools==75.8.0",
        "python -c \"import pkg_resources; print('pkg_resources OK:', pkg_resources.__file__)\"",

        # Build-time fix for RealESRGAN -> BasicSR on torchvision 0.20+.
        # BasicSR expects torchvision.transforms.functional_tensor, which was removed.
        # We create that module safely using chr(10), so no broken quoted newlines.
        """python - <<'PY'
from pathlib import Path
import site

roots = []
try:
    roots.extend(site.getsitepackages())
except Exception:
    pass
roots.append('/usr/local/lib/python3.10/site-packages')

seen = set()
for root in roots:
    if root in seen:
        continue
    seen.add(root)
    rootp = Path(root)

    tv_dir = rootp / 'torchvision' / 'transforms'
    if tv_dir.exists():
        shim = tv_dir / 'functional_tensor.py'
        nl = chr(10)
        shim.write_text(nl.join([
            'from torchvision.transforms.functional import *',
            'try:',
            '    from torchvision.transforms._functional_tensor import *',
            'except Exception:',
            '    pass',
            '',
        ]))
        print('Created torchvision functional_tensor shim:', shim)

    basicsr = rootp / 'basicsr'
    if basicsr.exists():
        for bp in basicsr.rglob('*.py'):
            txt = bp.read_text(errors='ignore')
            new_txt = txt.replace(
                'from torchvision.transforms.functional import rgb_to_grayscale',
                'from torchvision.transforms.functional_tensor import rgb_to_grayscale'
            )
            if new_txt != txt:
                bp.write_text(new_txt)
                print('Patched BasicSR file:', bp)
PY""",

        # Verify the exact import that was crashing before.
        """python - <<'PY'
import torchvision
import torchvision.transforms.functional_tensor as functional_tensor
from torchvision.transforms.functional_tensor import rgb_to_grayscale
print('torchvision:', torchvision.__version__)
print('functional_tensor shim OK:', functional_tensor.__file__)
print('rgb_to_grayscale OK:', rgb_to_grayscale)
PY""",

        "pip install bpy==4.0.0 --extra-index-url https://download.blender.org/pypi/ "
        "|| echo 'bpy optional: skipped'",

        f"cd {REPO_DIR}/hy3dpaint/custom_rasterizer && "
        f"LD_LIBRARY_PATH={LD_PATH} "
        "CUDA_HOME=/usr/local/cuda "
        "CUDACXX=/usr/local/cuda/bin/nvcc "
        "CC=gcc "
        "CXX=g++ "
        "MAX_JOBS=4 "
        "TORCH_CUDA_ARCH_LIST='8.0;8.6;8.9;9.0' "
        "pip install -e . --no-build-isolation --no-deps",

        f"cd {REPO_DIR}/hy3dpaint/DifferentiableRenderer && "
        f"LD_LIBRARY_PATH={LD_PATH} "
        "CUDA_HOME=/usr/local/cuda "
        "CUDACXX=/usr/local/cuda/bin/nvcc "
        "CC=gcc "
        "CXX=g++ "
        "MAX_JOBS=4 "
        "bash compile_mesh_painter.sh",

        f"mkdir -p {REPO_DIR}/hy3dpaint/ckpt && wget -nc "
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth "
        f"-P {REPO_DIR}/hy3dpaint/ckpt",

        # Hunyuan Paint may also look for ckpt/RealESRGAN_x4plus.pth relative to repo root.
        f"mkdir -p {REPO_DIR}/ckpt && ln -sf {REPO_DIR}/hy3dpaint/ckpt/RealESRGAN_x4plus.pth "
        f"{REPO_DIR}/ckpt/RealESRGAN_x4plus.pth && ls -l {REPO_DIR}/ckpt/RealESRGAN_x4plus.pth",

        "python -c \"import torch; print('final torch:', torch.__version__, torch.version.cuda)\"",
        f"LD_LIBRARY_PATH={LD_PATH} python -c \"import custom_rasterizer_kernel; print('custom_rasterizer ok')\"",
    )
    .env(
        {
            "HF_HOME": CACHE_DIR,
            "HF_HUB_CACHE": f"{CACHE_DIR}/hub",
            "TORCH_HOME": f"{CACHE_DIR}/torch",
            "HF_HUB_ENABLE_HF_TRANSFER": "1",
            "TEXT_TO_IMAGE_MODEL_ID": DEFAULT_T2I_MODEL,
            "CUDA_HOME": "/usr/local/cuda",
            "PATH": "/usr/local/cuda/bin:/usr/local/bin:/usr/bin:/bin",
            "LD_LIBRARY_PATH": LD_PATH,
            "CC": "gcc",
            "CXX": "g++",
            "CUDACXX": "/usr/local/cuda/bin/nvcc",
        }
    )
)

web_image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "fastapi[standard]==0.115.4",
    "pydantic==2.10.6",
)

app = modal.App(APP_NAME)


@app.cls(
    image=gpu_image,
    gpu=GPU_TYPE,
    timeout=2 * HOURS,
    scaledown_window=10 * MINUTES,
    volumes={CACHE_DIR: cache_volume, OUTPUT_DIR: outputs_volume},
    # secrets=[secret],
)
@modal.concurrent(max_inputs=1)
class Pipeline:
    @modal.enter()
    def start(self):
        import os
        import sys
        from pathlib import Path

        import torch

        os.makedirs(OUTPUT_DIR, exist_ok=True)
        os.makedirs(CACHE_DIR, exist_ok=True)
        os.chdir(REPO_DIR)

        sys.path.insert(0, REPO_DIR)
        sys.path.insert(0, str(Path(REPO_DIR) / "hy3dshape"))
        sys.path.insert(0, str(Path(REPO_DIR) / "hy3dpaint"))

        self.torch = torch
        self.sd_pipe = None
        self.shape_pipe = None
        self.paint_pipe = None
        self.paint_key = None

        self._load_sd()
        self._load_shape()
        print("[Pipeline] container ready")

    def _load_sd(self):
        if self.sd_pipe is not None:
            return self.sd_pipe

        import os
        from diffusers import AutoPipelineForText2Image

        model_id = os.environ.get("TEXT_TO_IMAGE_MODEL_ID", DEFAULT_T2I_MODEL)
        kwargs = dict(torch_dtype=self.torch.float16, use_safetensors=True)

        try:
            pipe = AutoPipelineForText2Image.from_pretrained(
                model_id,
                safety_checker=None,
                requires_safety_checker=False,
                **kwargs,
            )
        except TypeError:
            pipe = AutoPipelineForText2Image.from_pretrained(model_id, **kwargs)

        pipe = pipe.to("cuda")
        for opt in ("enable_attention_slicing", "enable_vae_slicing"):
            if hasattr(pipe, opt):
                getattr(pipe, opt)()

        self.sd_pipe = pipe
        return pipe

    def _load_shape(self):
        if self.shape_pipe is not None:
            return self.shape_pipe

        from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline

        try:
            self.shape_pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
                "tencent/Hunyuan3D-2.1",
                subfolder="hunyuan3d-dit-v2-1",
                torch_dtype=self.torch.float16,
            )
        except TypeError:
            self.shape_pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
                "tencent/Hunyuan3D-2.1"
            )

        return self.shape_pipe

    def _install_torchvision_functional_tensor_shim(self):
        """
        Runtime compatibility fix for:
        RealESRGAN -> BasicSR -> torchvision.transforms.functional_tensor
        New torchvision removed that module, so we register an in-memory module.
        """
        import sys
        import types

        module_name = "torchvision.transforms.functional_tensor"
        if module_name in sys.modules:
            return

        import torchvision.transforms.functional as functional

        shim = types.ModuleType(module_name)
        for name in dir(functional):
            if not name.startswith("__"):
                setattr(shim, name, getattr(functional, name))

        try:
            import torchvision.transforms._functional_tensor as internal_tensor
            for name in dir(internal_tensor):
                if not name.startswith("__"):
                    setattr(shim, name, getattr(internal_tensor, name))
        except Exception as exc:
            print(f"[Pipeline] _functional_tensor import skipped: {exc}")

        sys.modules[module_name] = shim
        print("[Pipeline] Installed torchvision.transforms.functional_tensor runtime shim")

    def _prepare_realesrgan_ckpt_paths(self):
        """
        Official Hunyuan Paint sometimes uses ckpt/RealESRGAN_x4plus.pth.
        The build downloads hy3dpaint/ckpt/RealESRGAN_x4plus.pth.
        This method guarantees both paths exist.
        """
        import os
        import shutil
        import subprocess
        from pathlib import Path

        repo = Path(REPO_DIR)
        real_ckpt = repo / "hy3dpaint" / "ckpt" / "RealESRGAN_x4plus.pth"
        compat_ckpt = repo / "ckpt" / "RealESRGAN_x4plus.pth"

        real_ckpt.parent.mkdir(parents=True, exist_ok=True)
        compat_ckpt.parent.mkdir(parents=True, exist_ok=True)

        if not real_ckpt.exists():
            print("[Pipeline] RealESRGAN ckpt missing, downloading...")
            subprocess.check_call([
                "wget",
                "-nc",
                "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
                "-P",
                str(real_ckpt.parent),
            ])

        if not compat_ckpt.exists():
            try:
                os.symlink(real_ckpt, compat_ckpt)
                print("[Pipeline] Created RealESRGAN compat symlink:", compat_ckpt, "->", real_ckpt)
            except Exception:
                shutil.copyfile(real_ckpt, compat_ckpt)
                print("[Pipeline] Copied RealESRGAN ckpt to compat path:", compat_ckpt)

        if not real_ckpt.exists():
            raise FileNotFoundError(f"Missing RealESRGAN ckpt: {real_ckpt}")
        if not compat_ckpt.exists():
            raise FileNotFoundError(f"Missing RealESRGAN compat ckpt: {compat_ckpt}")

        return real_ckpt, compat_ckpt

    def _load_paint(self, max_num_view: int, texture_resolution: int):
        """
        Load the REAL Hunyuan3D-Paint PBR pipeline using the same settings as
        your old working notebook code.
        """
        import os
        from pathlib import Path

        # Your old working PBR code used resolution=768.
        # Keep 768 for official PBR quality even if frontend sends 512.
        resolution = 768
        key = ("official_pbr", int(max_num_view), resolution)

        if self.paint_pipe is not None and self.paint_key == key:
            return self.paint_pipe

        self.torch.cuda.empty_cache()
        os.chdir(REPO_DIR)

        self._install_torchvision_functional_tensor_shim()

        try:
            import pkg_resources
            print("[Pipeline] pkg_resources OK:", pkg_resources.__file__)
        except Exception as exc:
            raise RuntimeError("pkg_resources is missing. setuptools==75.8.0 must be installed.") from exc

        real_ckpt, compat_ckpt = self._prepare_realesrgan_ckpt_paths()

        import torchvision
        print("[Pipeline] torchvision OK:", torchvision.__version__)
        print("[Pipeline] RealESRGAN real ckpt:", real_ckpt)
        print("[Pipeline] RealESRGAN compat ckpt:", compat_ckpt)

        from textureGenPipeline import Hunyuan3DPaintConfig, Hunyuan3DPaintPipeline

        conf = Hunyuan3DPaintConfig(max_num_view=int(max_num_view), resolution=resolution)

        # EXACT PBR settings from your old working code.
        # We keep relative paths because the official code is usually written around cd REPO_DIR.
        conf.realesrgan_ckpt_path = "hy3dpaint/ckpt/RealESRGAN_x4plus.pth"
        conf.multiview_cfg_path = "hy3dpaint/cfgs/hunyuan-paint-pbr.yaml"
        conf.custom_pipeline = "hy3dpaint/hunyuanpaintpbr"

        # Safety checks.
        if not Path(conf.realesrgan_ckpt_path).exists():
            raise FileNotFoundError(f"RealESRGAN ckpt missing: {conf.realesrgan_ckpt_path}")
        if not Path("ckpt/RealESRGAN_x4plus.pth").exists():
            raise FileNotFoundError("RealESRGAN compatibility ckpt missing: ckpt/RealESRGAN_x4plus.pth")
        if not Path(conf.multiview_cfg_path).exists():
            raise FileNotFoundError(f"PBR cfg missing: {conf.multiview_cfg_path}")
        if not Path(conf.custom_pipeline).exists():
            raise FileNotFoundError(f"PBR custom pipeline missing: {conf.custom_pipeline}")

        print("[Pipeline] Loading OFFICIAL Hunyuan3D-Paint PBR pipeline...")
        print("[Pipeline] cfg:", conf.multiview_cfg_path)
        print("[Pipeline] custom_pipeline:", conf.custom_pipeline)
        print("[Pipeline] realesrgan:", conf.realesrgan_ckpt_path)

        self.paint_pipe = Hunyuan3DPaintPipeline(conf)
        self.paint_key = key
        return self.paint_pipe

    def _generate_sd_image(self, prompt, out_path, seed, width, height, steps, guidance):
        pipe = self._load_sd()

        full_prompt = (
            "single centered 3D asset reference image, full object visible, clean "
            "silhouette, plain white background, product render, high detail, " + prompt
        )
        negative = (
            "cropped, cut off, multiple objects, extra objects, text, watermark, logo, "
            "blurry, low quality, busy background, hands, people unless requested"
        )

        generator = None
        if seed is not None:
            generator = self.torch.Generator(device="cuda").manual_seed(int(seed))

        with self.torch.inference_mode():
            image = pipe(
                prompt=full_prompt,
                negative_prompt=negative,
                width=int(width),
                height=int(height),
                num_inference_steps=int(steps),
                guidance_scale=float(guidance),
                generator=generator,
            ).images[0]

        image.save(out_path)
        self.torch.cuda.empty_cache()

    @staticmethod
    def _remove_background(in_path, out_path):
        import shutil
        from PIL import Image

        try:
            from rembg import remove

            out = remove(Image.open(in_path).convert("RGBA"))
            out.save(out_path)
        except Exception as exc:
            print(f"[Pipeline] rembg skipped ({exc}); using original image")
            shutil.copyfile(in_path, out_path)

    @staticmethod
    def _export_mesh(mesh_result, out_glb):
        import shutil
        from pathlib import Path

        if isinstance(mesh_result, (str, Path)):
            src = Path(mesh_result)
            if src.suffix.lower() == ".glb":
                shutil.copyfile(src, out_glb)
                return

            import trimesh

            trimesh.load(str(src), force="mesh").export(str(out_glb))
            return

        if isinstance(mesh_result, (list, tuple)):
            mesh_result = mesh_result[0]

        if hasattr(mesh_result, "export"):
            mesh_result.export(str(out_glb))
            return

        raise RuntimeError(f"Unsupported Hunyuan shape output type: {type(mesh_result)!r}")

    def _generate_shape(self, condition_image, raw_glb):
        result = self._load_shape()(image=str(condition_image))
        self._export_mesh(result, raw_glb)
        self.torch.cuda.empty_cache()

    def _write_texture_debug(self, debug_path, payload):
        import json
        debug_path.write_text(json.dumps(payload, indent=2))

    def _apply_texture(self, raw_glb, condition_image, textured_glb, max_num_view, texture_resolution):
        """
        Official Hunyuan3D-Paint PBR texture generation only.
        No fake fallback here. If official PBR fails, the job fails and writes
        texture_debug.json with the real error.
        """
        import os
        import shutil
        from pathlib import Path

        old_cwd = os.getcwd()
        os.chdir(REPO_DIR)

        job_dir = raw_glb.parent
        debug_path = job_dir / "texture_debug.json"

        payload = {
            "mode": "official_hunyuan_paint_pbr",
            "raw_glb": str(raw_glb),
            "condition_image": str(condition_image),
            "requested_output": str(textured_glb),
            "returned": None,
            "selected_glb": None,
            "texture_files": [],
        }

        try:
            paint = self._load_paint(max_num_view, texture_resolution)

            print("[Pipeline] Generating OFFICIAL Hunyuan3D-Paint PBR texture...")
            returned = paint(
                mesh_path=str(raw_glb),
                image_path=str(condition_image),
                output_mesh_path=str(textured_glb),
            )

            self.torch.cuda.empty_cache()
            payload["returned"] = str(returned) if returned else None

            all_files = [x for x in job_dir.rglob("*") if x.is_file()]
            payload["after_files"] = sorted([str(x.relative_to(job_dir)) for x in all_files])

            # Expose PBR texture image files, if the official pipeline writes any sidecar textures.
            image_exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tga", ".exr"}
            texture_candidates = [
                x for x in all_files
                if x.suffix.lower() in image_exts
                and x.name not in {"condition.png", "sd_condition.png", "texture.png"}
            ]
            texture_candidates = sorted(texture_candidates, key=lambda x: x.stat().st_mtime, reverse=True)

            for i, tex in enumerate(texture_candidates):
                target_name = "texture.png" if i == 0 and tex.suffix.lower() != ".exr" else f"texture_{i}{tex.suffix.lower()}"
                target = job_dir / target_name
                try:
                    if tex.resolve() != target.resolve():
                        shutil.copyfile(tex, target)
                    payload["texture_files"].append(target.name)
                except Exception as exc:
                    payload.setdefault("texture_copy_errors", []).append({"source": str(tex), "error": str(exc)})

            # Find the official textured GLB.
            candidates = []
            if returned:
                r = Path(str(returned))
                candidates.append(r)
                if r.suffix.lower() != ".glb":
                    candidates.append(r.with_suffix(".glb"))

            candidates.append(textured_glb)

            for glb in job_dir.rglob("*.glb"):
                if glb.name != raw_glb.name:
                    candidates.append(glb)

            seen = set()
            candidates = [x for x in candidates if not (str(x) in seen or seen.add(str(x)))]

            for cand in candidates:
                if cand.exists() and cand.suffix.lower() == ".glb" and cand.stat().st_size > 1000:
                    if cand.resolve() != textured_glb.resolve():
                        shutil.copyfile(cand, textured_glb)
                    payload["selected_glb"] = str(cand)
                    payload["final_textured_glb"] = textured_glb.name
                    self._write_texture_debug(debug_path, payload)
                    print("[Pipeline] OFFICIAL PBR texture done:", textured_glb)
                    return

            self._write_texture_debug(debug_path, payload)
            raise RuntimeError("Official Hunyuan PBR finished but no textured GLB was found.")

        except Exception as exc:
            payload["error"] = f"{type(exc).__name__}: {exc}"
            self._write_texture_debug(debug_path, payload)
            raise

        finally:
            os.chdir(old_cwd)

    @modal.method()
    def run(self, job_id: str, params: dict) -> dict:
        import base64
        import shutil
        import traceback
        from pathlib import Path

        def set_status(**fields):
            entry = job_store.get(job_id, {})
            entry.setdefault("job_id", job_id)
            entry.update(fields)
            job_store[job_id] = entry

        set_status(
            status="running",
            progress=3,
            stage="starting",
            message="Starting AI generation...",
        )

        job_dir = Path(OUTPUT_DIR) / job_id
        job_dir.mkdir(parents=True, exist_ok=True)

        generated = job_dir / "sd_condition.png"
        condition = job_dir / "condition.png"
        raw_glb = job_dir / "mesh_untextured.glb"
        textured_glb = job_dir / "mesh_textured.glb"
        model_glb = job_dir / "model.glb"
        debug_json = job_dir / "texture_debug.json"

        try:
            image_b64 = params.get("image_base64")
            prompt = params.get("prompt") or params.get("voice_text")

            if image_b64:
                set_status(
                    progress=10,
                    stage="image_received",
                    message="Reading uploaded image...",
                )
                data = image_b64
                if "," in data and data.split(",", 1)[0].startswith("data:"):
                    data = data.split(",", 1)[1]
                generated.write_bytes(base64.b64decode(data))
                set_status(
                    progress=25,
                    stage="image_ready",
                    message="Input image ready.",
                )
            elif prompt:
                set_status(
                    progress=10,
                    stage="text_to_image",
                    message="Generating reference image...",
                )
                self._generate_sd_image(
                    prompt,
                    generated,
                    params.get("seed"),
                    params.get("width", 512),
                    params.get("height", 512),
                    params.get("sd_steps", 28),
                    params.get("guidance_scale", 7.0),
                )
                set_status(
                    progress=30,
                    stage="image_ready",
                    message="Reference image generated.",
                )
            else:
                raise ValueError("Provide prompt / voice_text or image_base64")

            set_status(
                progress=35,
                stage="background_removal",
                message="Preparing clean object mask...",
            )
            if params.get("remove_background", True):
                self._remove_background(generated, condition)
            else:
                shutil.copyfile(generated, condition)

            set_status(
                progress=42,
                stage="condition_ready",
                message="Condition image ready.",
            )

            set_status(
                progress=45,
                stage="image_to_mesh",
                message="Creating 3D mesh...",
            )
            self._generate_shape(condition, raw_glb)

            set_status(
                progress=65,
                stage="mesh_ready",
                message="3D mesh created.",
            )

            if params.get("with_texture", True):
                set_status(
                    progress=70,
                    stage="texturing",
                    message="Applying official PBR texture...",
                )
                self._apply_texture(
                    raw_glb,
                    condition,
                    textured_glb,
                    params.get("max_num_view", 6),
                    params.get("texture_resolution", 512),
                )
                set_status(
                    progress=90,
                    stage="texture_ready",
                    message="Texture completed.",
                )
                final_glb = textured_glb
                final_kind = "textured"
            else:
                set_status(
                    progress=85,
                    stage="mesh_only_ready",
                    message="Mesh-only model ready.",
                )
                final_glb = raw_glb
                final_kind = "untextured"

            set_status(
                progress=95,
                stage="finalizing",
                message="Finalizing 3D model...",
            )

            # Always create model.glb because your frontend expects:
            # /download/{job_id}/model.glb
            shutil.copyfile(final_glb, model_glb)

            files = {
                "final_glb": model_glb.name,
                "model_glb": model_glb.name,
                "raw_glb": raw_glb.name,
                "condition_image": condition.name,
            }

            # Surface any visible texture/material sidecar files in Modal storage / API.
            texture_sidecars = []
            for sidecar in sorted(job_dir.glob("*")):
                if sidecar.is_file() and sidecar.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tga", ".exr", ".mtl", ".obj"}:
                    if sidecar.name not in {generated.name, condition.name}:
                        texture_sidecars.append(sidecar.name)

            if texture_sidecars:
                files["texture_files"] = texture_sidecars
                for name in texture_sidecars:
                    if name.lower().endswith((".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tga", ".exr")):
                        files["texture_png"] = name
                        break

            if debug_json.exists():
                files["texture_debug"] = debug_json.name

            metadata = {
                "job_id": job_id,
                "params": params,
                "files": files,
                "final_kind": final_kind,
            }
            (job_dir / "metadata.json").write_text(json.dumps(metadata, indent=2))
            outputs_volume.commit()

            set_status(
                status="succeeded",
                progress=100,
                stage="done",
                message="3D model ready.",
                with_texture=bool(params.get("with_texture", True)),
                final_kind=final_kind,
                files=files,
            )
            return {
                "job_id": job_id,
                "status": "succeeded",
                "progress": 100,
                "stage": "done",
                "message": "3D model ready.",
                "files": files,
                "final_kind": final_kind,
            }

        except Exception as exc:
            traceback.print_exc()
            try:
                outputs_volume.commit()
            except Exception:
                pass

            entry = {
                "status": "failed",
                "stage": "failed",
                "message": "Generation failed.",
                "error": f"{type(exc).__name__}: {exc}",
            }
            if debug_json.exists():
                entry["files"] = {"texture_debug": debug_json.name}
            set_status(**entry)
            return {
                "job_id": job_id,
                "status": "failed",
                "stage": "failed",
                "message": "Generation failed.",
                "error": str(exc),
            }


@app.function(
    image=web_image,
    volumes={OUTPUT_DIR: outputs_volume},
    timeout=5 * MINUTES,
    # secrets=[secret],
)
@modal.concurrent(max_inputs=100)
@modal.asgi_app()
def api():
    import base64
    import mimetypes
    import os
    import uuid
    from pathlib import Path
    from typing import Optional

    from fastapi import FastAPI, File, Form, HTTPException, Request, UploadFile
    from fastapi.middleware.cors import CORSMiddleware
    from fastapi.responses import FileResponse, JSONResponse
    from pydantic import BaseModel, Field

    web = FastAPI(title="R2V SD + Hunyuan3D API", version="2.8.0-official-pbr-pkgresources")

    origins = [o.strip() for o in os.environ.get("R2V_CORS_ORIGINS", "*").split(",") if o.strip()]
    web.add_middleware(
        CORSMiddleware,
        allow_origins=origins or ["*"],
        allow_credentials=False,
        allow_methods=["*"],
        allow_headers=["*"],
    )

    def check_auth(request: Request) -> None:
        expected = os.environ.get("R2V_API_TOKEN")
        if expected and request.headers.get("authorization", "") != f"Bearer {expected}":
            raise HTTPException(status_code=401, detail="Invalid or missing bearer token")

    def safe_name(name: str, fallback: str) -> str:
        name = (name or fallback).replace("\\", "_").replace("/", "_").strip()
        return name or fallback

    def submit(params: dict) -> dict:
        job_id = str(uuid.uuid4())
        job_store[job_id] = {
            "job_id": job_id,
            "status": "queued",
            "progress": 0,
            "stage": "queued",
            "message": "Queued for generation...",
        }
        call = Pipeline().run.spawn(job_id, params)

        entry = job_store[job_id]
        entry["call_id"] = call.object_id
        job_store[job_id] = entry

        return {
            "job_id": job_id,
            "status": "queued",
            "progress": 0,
            "stage": "queued",
            "message": "Queued for generation...",
            "poll_url": f"/jobs/{job_id}",
        }

    class GenerateRequest(BaseModel):
        prompt: Optional[str] = Field(default=None, description="Text prompt.")
        voice_text: Optional[str] = Field(default=None, description="Voice transcript.")
        image_base64: Optional[str] = Field(default=None, description="Optional base64 image.")
        with_texture: bool = Field(default=True, description="True = textured, False = mesh only.")
        seed: Optional[int] = None
        width: int = Field(default=512, ge=384, le=1024)
        height: int = Field(default=512, ge=384, le=1024)
        sd_steps: int = Field(default=28, ge=1, le=80)
        guidance_scale: float = Field(default=7.0, ge=0.0, le=20.0)
        max_num_view: int = Field(default=6, ge=6, le=9)
        texture_resolution: int = Field(default=512, description="512 or 768")
        remove_background: bool = Field(default=True)

    @web.get("/")
    async def root():
        return {
            "ok": True,
            "version": "2.6.0-functional-tensor-shim",
            "name": "R2V SD + Hunyuan3D API",
            "workflow": "prompt/voice/image -> condition image -> mesh -> optional texture -> GLB",
            "routes": [
                "GET /health",
                "POST /generate",
                "POST /text-to-3d",
                "POST /prompt-to-3d",
                "POST /image-to-3d",
                "GET /jobs/{job_id}",
                "GET /download/{job_id}/{filename}",
            ],
        }

    @web.get("/health")
    async def health():
        return {"ok": True}

    @web.post("/generate")
    @web.post("/text-to-3d")
    @web.post("/prompt-to-3d")
    async def generate(req: GenerateRequest, request: Request):
        check_auth(request)
        if not (req.prompt or req.voice_text or req.image_base64):
            raise HTTPException(status_code=400, detail="Send prompt, voice_text, or image_base64")
        return JSONResponse(submit(req.model_dump()))

    @web.post("/image-to-3d")
    async def image_to_3d(
        request: Request,
        file: UploadFile = File(...),
        with_texture: bool = Form(True),
        max_num_view: int = Form(6),
        texture_resolution: int = Form(512),
        remove_background: bool = Form(True),
    ):
        check_auth(request)
        raw = await file.read()
        params = {
            "image_base64": base64.b64encode(raw).decode("utf-8"),
            "with_texture": with_texture,
            "max_num_view": max_num_view,
            "texture_resolution": texture_resolution,
            "remove_background": remove_background,
        }
        return JSONResponse(submit(params))

    @web.get("/jobs/{job_id}")
    async def job_status(job_id: str, request: Request):
        entry = job_store.get(job_id)
        if entry is None:
            raise HTTPException(status_code=404, detail="Unknown job_id")

        out = dict(entry)
        out.setdefault("job_id", job_id)
        out.setdefault("progress", 0)
        out.setdefault("stage", out.get("status", "unknown"))
        out.setdefault("message", "")
        files = out.get("files", {})
        base = str(request.base_url).rstrip("/")

        if out.get("status") == "succeeded":
            final = files.get("final_glb")
            if final:
                out["glb_url"] = f"{base}/download/{job_id}/{final}"
                out["model_url"] = out["glb_url"]
                out["download_url"] = out["glb_url"]

            if files.get("raw_glb"):
                out["raw_glb_url"] = f"{base}/download/{job_id}/{files.get('raw_glb')}"

            if files.get("condition_image"):
                out["condition_image_url"] = f"{base}/download/{job_id}/{files.get('condition_image')}"

            if files.get("texture_png"):
                out["texture_png_url"] = f"{base}/download/{job_id}/{files.get('texture_png')}"

            if files.get("texture_debug"):
                out["texture_debug_url"] = f"{base}/download/{job_id}/{files.get('texture_debug')}"

            if files.get("texture_files"):
                out["texture_file_urls"] = [
                    f"{base}/download/{job_id}/{name}" for name in files.get("texture_files", [])
                ]

            out["artifacts"] = {
                "glb_url": out.get("glb_url"),
                "model_url": out.get("model_url"),
                "raw_glb_url": out.get("raw_glb_url"),
                "condition_image_url": out.get("condition_image_url"),
                "texture_debug_url": out.get("texture_debug_url"),
                "texture_file_urls": out.get("texture_file_urls"),
            }

        elif out.get("status") == "failed":
            if files.get("texture_debug"):
                out["texture_debug_url"] = f"{base}/download/{job_id}/{files.get('texture_debug')}"

        return JSONResponse(out)

    @web.get("/download/{job_id}/{filename}")
    async def download(job_id: str, filename: str):
        try:
            outputs_volume.reload()
        except Exception:
            pass

        filename = safe_name(filename, "file.bin")
        path = Path(OUTPUT_DIR) / safe_name(job_id, "job") / filename

        if not path.exists():
            raise HTTPException(status_code=404, detail="File not found")

        media = mimetypes.guess_type(path.name)[0]
        if path.suffix.lower() == ".glb":
            media = "model/gltf-binary"
        elif path.suffix.lower() == ".json":
            media = "application/json"

        return FileResponse(path, media_type=media or "application/octet-stream", filename=path.name)

    return web


@app.local_entrypoint()
def prewarm():
    print("Warming model cache...")
    Pipeline().run.remote(
        "prewarm",
        {
            "prompt": "a simple grey cube, single object, white background",
            "with_texture": False,
            "sd_steps": 8,
            "seed": 0,
        },
    )
    print("Done.")




Writing r2v_modal_app.py


In [2]:
# 2. Validate the app file compiles before deploying.
import py_compile
py_compile.compile("r2v_modal_app.py", doraise=True)
print("Syntax OK: r2v_modal_app.py")


Syntax OK: r2v_modal_app.py


In [3]:
# 3. Deploy to Modal.
# This is the main deploy cell. First deploy builds the CUDA image and compiles
# Hunyuan3D texture renderers, so it can take time. Later deploys are cached.
!modal deploy r2v_modal_app.py


⠸ Creating objects...
⠦ Creating objects...
├── 🔨 Created mount /root/r2v_modal_app.py
⠏ Creating objects...
├── 🔨 Created mount /root/r2v_modal_app.py
⠹ Creating objects...
├── 🔨 Created mount /root/r2v_modal_app.py
├── 🔨 Created web function api => https://ma1260781--r2v-sd-hunyuan3d-api.modal.run
⠴ Creating objects...
├── 🔨 Created mount /root/r2v_modal_app.py
├── 🔨 Created web function api => https://ma1260781--r2v-sd-hunyuan3d-api.modal.run
└── 🔨 Created function Pipeline.*.
✓ Created objects.
├── 🔨 Created mount /root/r2v_modal_app.py
├── 🔨 Created web function api => https://ma1260781--r2v-sd-hunyuan3d-api.modal.run
└── 🔨 Created function Pipeline.*.
✓ App deployed in 1.922s! 🎉

View Deployment: https://modal.com/apps/ma1260781/main/deployed/r2v-sd-hunyuan3d


In [5]:
import requests
MODAL_URL = "https://ma056612--r2v-sd-hunyuan3d-api.modal.run"
r = requests.get(f"{MODAL_URL}/health", timeout=60)
print(r.status_code, r.text)

404 modal-http: invalid function call



In [2]:
import requests
import time

MODAL_URL = "https://ma1260781--r2v-sd-hunyuan3d-api.modal.run"

r = requests.post(
    f"{MODAL_URL}/generate",
    json={
        "prompt": "a red sports sneaker, high quality 3d asset",
        "with_texture": True,
        "sd_steps": 20,
        "seed": 123
    },
    timeout=60
)

print(r.status_code, r.text)

job_id = r.json()["job_id"]
print("JOB:", job_id)

while True:
    s = requests.get(f"{MODAL_URL}/jobs/{job_id}", timeout=60).json()

    print(
        s.get("status"),
        s.get("progress"),
        s.get("stage"),
        "-",
        s.get("message")
    )

    if s.get("status") in ["succeeded", "failed"]:
        print(s)
        break

    time.sleep(2)

200 {"job_id":"0a1035d0-a13f-4a1b-98ef-f9121d5e0ee0","status":"queued","progress":0,"stage":"queued","message":"Queued for generation...","poll_url":"/jobs/0a1035d0-a13f-4a1b-98ef-f9121d5e0ee0"}
JOB: 0a1035d0-a13f-4a1b-98ef-f9121d5e0ee0
queued 0 queued - Queued for generation...
queued 0 queued - Queued for generation...
queued 0 queued - Queued for generation...
queued 0 queued - Queued for generation...
queued 0 queued - Queued for generation...
queued 0 queued - Queued for generation...
queued 0 queued - Queued for generation...
queued 0 queued - Queued for generation...
queued 0 queued - Queued for generation...
queued 0 queued - Queued for generation...
queued 0 queued - Queued for generation...
queued 0 queued - Queued for generation...
queued 0 queued - Queued for generation...
queued 0 queued - Queued for generation...
queued 0 queued - Queued for generation...
queued 0 queued - Queued for generation...
queued 0 queued - Queued for generation...
queued 0 queued - Queued for gen

## 3. API reference

**Text / voice → 3D** — `POST /generate`
```json
{
  "prompt": "a red sports sneaker",
  "voice_text": null,
  "with_texture": true,
  "seed": 123,
  "texture_resolution": 512,
  "max_num_view": 6
}
```
Mesh only (faster): set `"with_texture": false`.

**Image → 3D** — `POST /image-to-3d` (multipart form): `file` plus optional `with_texture`, `max_num_view`, `texture_resolution`, `remove_background`.

**Submit response** (returns instantly):
```json
{ "job_id": "…", "status": "queued", "poll_url": "/jobs/…" }
```

**`GET /jobs/{job_id}`** while running:
```json
{ "status": "running", "stage": "texturing" }
```
On success it adds `glb_url`, `model_url`, `download_url`, `raw_glb_url`, `condition_image_url`, and an `artifacts` object — so your existing adapter that looks for `glb_url` / `model_url` / `download_url` keeps working.


## 4. Backend integration (async)

Frontend sends the texture choice in `AIJob.settings_json`: `{"with_texture": true}` for **Yes, texture**, `false` for **No, mesh only**.

Because generation is now asynchronous, the worker submits then polls:

```python
import time, requests

def submit_prompt(api_url, prompt, *, with_texture=True, token=None):
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    r = requests.post(f"{api_url}/generate",
                      json={"prompt": prompt, "with_texture": bool(with_texture)},
                      headers=headers, timeout=60)
    r.raise_for_status()
    return r.json()["job_id"], headers

def wait_for_glb(api_url, job_id, headers, out_glb, *, poll=10, max_wait=2*60*60):
    deadline = time.time() + max_wait
    while time.time() < deadline:
        s = requests.get(f"{api_url}/jobs/{job_id}", headers=headers, timeout=60).json()
        if s["status"] == "succeeded":
            data = requests.get(s["glb_url"], headers=headers, timeout=600).content
            open(out_glb, "wb").write(data)
            return out_glb
        if s["status"] == "failed":
            raise RuntimeError(s.get("error", "generation failed"))
        time.sleep(poll)
    raise TimeoutError("generation timed out")
```

In `tasks.py`:
```python
settings = job.settings_json or {}
with_texture = bool(settings.get("with_texture", settings.get("texture", True)))
job_id, headers = submit_prompt(MODAL_API_URL, job.prompt, with_texture=with_texture, token=R2V_API_TOKEN)
wait_for_glb(MODAL_API_URL, job_id, headers, glb_raw)
```
For an uploaded image, post to `/image-to-3d` with `files={"file": ...}` and `data={"with_texture": "true"}` instead.


## 5. Config, auth & troubleshooting

**Switch model / GPU** — edit the top of `r2v_modal_app.py`:
- `GPU_TYPE` — `"H100"` (recommended) or `"A100-80GB"`. Texturing is VRAM-heavy; 40GB cards may OOM at `texture_resolution=768`.
- `DEFAULT_T2I_MODEL` — keep the SD1.5 mirror, or set `"stabilityai/stable-diffusion-xl-base-1.0"` for sharper condition images (uses more VRAM).

**Enable auth + gated models** (optional):
```bash
modal secret create r2v-secrets R2V_API_TOKEN=choose-a-strong-token HF_TOKEN=hf_xxx
```
Then uncomment the `secret = modal.Secret.from_name("r2v-secrets")` line and the `secrets=[secret]` lines in the app, and redeploy. When `R2V_API_TOKEN` is set, every request needs `Authorization: Bearer <token>`.

**Lock CORS** to your site: set `R2V_CORS_ORIGINS=https://yourapp.com` in the secret (defaults to `*`).

**Common issues**
- *First request slow* — run cell 4 (`modal run`) once to pre-download weights.
- *Texture step OOMs* — drop `texture_resolution` to 512, lower `max_num_view`, or use an 80GB GPU.
- *401 Unauthorized* — you set a token; send the `Authorization` header.
- *Job stuck at `queued`* — check `modal app logs r2v-sd-hunyuan3d`; the GPU container may still be building/booting.

> Note: this notebook was statically validated (syntax + structure + helper logic). The model pipeline itself can only be exercised on a real Modal GPU, so do a first run with cell 5 after deploying.
